# Data Exploration: Shopping Cart Trajectories

In this notebook, we explore the shopping cart data located in `../data/raw`. The data consists of `x`, `y` coordinates, timestamps, and a quality metric `q`.

Goals:
- Explore raw data quality and drift.
- Filter out artifacts (charging stations, outside shop bounds) to assess data usability.
- Re-analyze drift on the cleaned data.
- Assess positioning accuracy on stationary devices (standard deviation).

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style('whitegrid')

In [ ]:
# Load a sample of the data 
raw_data_dir = '../data/raw'
all_files = glob.glob(os.path.join(raw_data_dir, 'node_*.csv'))

import gc
dfs = []
for f in all_files[:3]: # Load 3 files minimum to see drift vs q but save memory
    df = pd.read_csv(f, usecols=['node_id', 'timestamp', 'x', 'y', 'q'], dtype={'node_id': 'int32', 'x': 'float32', 'y': 'float32', 'q': 'uint8'})
    dfs.append(df)
    
data = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()
data['timestamp'] = pd.to_datetime(data['timestamp'])
data = data.sort_values(by=['node_id', 'timestamp'])
print("Raw Data Shape:", data.shape)

## 1. Raw Data Exploration (Before Filtering)
Let's check for missing values, the distribution of `q` (quality), and the drift on raw data.

In [ ]:
print("Missing values:\n", data.isnull().sum())
print("\nData Describe:\n", data.describe())

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['q'], bins=50, kde=True)
plt.title('Distribution of Quality (q) - Raw Data')
plt.xlabel('q')
plt.show()

In [ ]:
# Calculate time difference and distance between consecutive points for each node
data['time_diff'] = data.groupby('node_id')['timestamp'].diff().dt.total_seconds()
data['x_diff'] = data.groupby('node_id')['x'].diff()
data['y_diff'] = data.groupby('node_id')['y'].diff()
data['distance'] = np.sqrt(data['x_diff']**2 + data['y_diff']**2)

# Calculate speed (units per second) - approximating drift
data['speed'] = data['distance'] / data['time_diff']

plt.figure(figsize=(10, 5))
sns.histplot(data[data['distance'] < data['distance'].quantile(0.95)]['distance'], bins=50)
plt.title('Distribution of Distance between Consecutive Points (Raw Data, Zoomed to 95th Percentile)')
plt.xlabel('Distance')
plt.show()

In [ ]:
# Raw Drift vs Q
plt.figure(figsize=(10, 6))
sns.scatterplot(x='q', y='distance', data=data, alpha=0.3)
plt.title('Raw Data: Scatter Plot of Quality (q) vs Distance (Drift)')
plt.xlabel('Quality (q)')
plt.ylabel('Distance')
plt.yscale('log') 
plt.show()

## 2. Positioning Usability (Hyvyys/Käytettävyys) - Applying Filters
We count how many points are actual shop visits versus artifacts (out of bounds or charging stations without signal at 0,0).

In [ ]:
data['is_charging_station'] = (data['x'] == 0) & (data['y'] == 0)

# Määritellään kaupan fyysiset koordinaattirajat
shop_bounds = (
    ((data['x'] > 350) & (data['y'] < 3000) & (data['x'] < 1500)) |
    ((data['x'] > 1500) & (data['x'] < 8200)) |
    ((data['x'] > 8200) & (data['y'] > 450) & (data['x'] < 9650)) |
    ((data['x'] > 9650) & (data['y'] > 450) & (data['y'] < 4700) & (data['x'] < 10190))
)
data['in_shop'] = shop_bounds
data['is_out_of_bounds'] = ~data['in_shop'] & ~data['is_charging_station']

in_shop_pts = data['in_shop'].sum()
charging_pts = data['is_charging_station'].sum()
out_pts = data['is_out_of_bounds'].sum()

print(f"Total Points: {len(data)}")
print(f"- Inside Shop (Usable): {in_shop_pts}")
print(f"- Charging Station (0,0): {charging_pts}")
print(f"- Outside Shop (Noise): {out_pts}")

labels = ['Inside Shop', 'Charging Station (0,0)', 'Outside Bounds (Noise)']
sizes = [in_shop_pts, charging_pts, out_pts]
colors = ['#4CAF50', '#FFC107', '#F44336']

plt.figure(figsize=(7, 7))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=140)
plt.title('Positioning Usability')
plt.show()

## 3. Drift Analysis on Cleaned Data
Let's check the drift vs Q on ONLY the cleaned in-shop data, removing the massive leaps caused by out-of-bounds metrics.

In [ ]:
clean_data = data[data['in_shop']].copy()

clean_data['q_bin'] = pd.qcut(clean_data['q'], q=5, duplicates='drop')
plt.figure(figsize=(10, 6))
sns.boxplot(x='q_bin', y='distance', data=clean_data)
plt.title('Cleaned Data: Boxplot of Distance (Drift) by Quality (q) Bins')
plt.xlabel('q Bins')
plt.ylabel('Distance')
plt.yscale('log')
plt.show()

## 4. Positioning Accuracy using Stationary Devices (Paikannustarkkuus)
We identify periods when the device is stationary (e.g. speed < 5) and calculate the standard deviation of `x` and `y`.

In [ ]:
stationary_data = clean_data[clean_data['speed'] < 5.0].copy()

print(f"Number of Stationary Points: {len(stationary_data)}")
print(f"X Standard Deviation (Accuracy): {stationary_data['x_diff'].std():.2f}")
print(f"Y Standard Deviation (Accuracy): {stationary_data['y_diff'].std():.2f}")
print(f"Average Euclidean Error when stationary: {stationary_data['distance'].mean():.2f}")

# Check standard deviation vs Q
stationary_data['q_rounded'] = stationary_data['q'].round(-1)
std_by_q = stationary_data.groupby('q_rounded').agg({'x_diff': 'std', 'y_diff': 'std'}).reset_index()

plt.figure(figsize=(10, 5))
plt.plot(std_by_q['q_rounded'], std_by_q['x_diff'], label='X Standard Deviation', marker='o')
plt.plot(std_by_q['q_rounded'], std_by_q['y_diff'], label='Y Standard Deviation', marker='s')
plt.title('Positioning Standard Deviation vs Quality (q) [Stationary Devices]')
plt.xlabel('Quality (q)')
plt.ylabel('Standard Deviation (Distance)')
plt.legend()
plt.show()